### Understanding Wrapper style for model

In [ ]:
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_ollama import ChatOllama
from langchain.tools import tool
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.messages import SystemMessage
from pydantic import BaseModel
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse

@tool
def get_weather_city(city: str) -> str:
    """Get the current weather for a city
    
    Args:
        city: The name of the city for which to get weather information
    
    Returns:
        A string describing the weather in the specified city
    """
    return f"The weather in {city} is sunny."

#model
basic_model = ChatOllama(
    base_url="https://ollama.com",
    model="qwen3-coder-next:cloud",
    client_kwargs=  {  
        "headers": {'Authorization': 'Bearer ' + '5172773b797243f6939e3f34642fbe4a.iahgs1TfHbkk9PBtNE4wY5a2'}
        }
    ) 

advanced_model = ChatOllama(
    base_url="https://ollama.com",
    model="qwen3.5:397b-cloud",
    client_kwargs=  {  
        "headers": {'Authorization': 'Bearer ' + '5172773b797243f6939e3f34642fbe4a.iahgs1TfHbkk9PBtNE4wY5a2'}
        }
    ) 

#Choose Model dynamically
@wrap_model_call
def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:
    print("dynamic_model_selection middleware invoked!")
    msg:str = request.messages[0].content
    dynamic_model = request.model
    if  msg.find("simple")< 0:
         print("in if")
         dynamic_model = advanced_model
    else: 
        print("in else")
        dynamic_model = request.model

    res = handler(request.override(model=dynamic_model))
    print("dynamic_model_selection middleware Done!")
    print(res)
    return res

@wrap_model_call
def dynamic_tool_selection(request: ModelRequest, handler) -> ModelResponse:    
    print("dynamic_tool_selection middleware invoked!")
    tool_list = request.tools
    msg:str = request.messages[0].content
    dynamic_model = request.model
    if  msg.find("weather")< 0:
         print("in if")
         tool_list = []
    else: 
        print("in else")
        tool_list = [get_weather_city]

    res = handler(request.override(tools=tool_list))
    print("dynamic_tool_selection middleware Done!")
    print(res)
    return res


tool_list = [get_weather_city]

agent = create_agent(
    name= "PodTest Agent",
    model=basic_model,
    tools=tool_list,
    middleware=[dynamic_model_selection, dynamic_tool_selection]   
)


prompt = PromptTemplate.from_template("What is the weather in Delhi give me advanced answer?")
promptValue = prompt.invoke({})

respo = agent.invoke({"messages": str(promptValue)})
#print(respo)






dynamic_model_selection middleware invoked!
in if
dynamic_tool_selection middleware invoked!
in else
dynamic_tool_selection middleware Done!
ModelResponse(result=[AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.5:397b-cloud', 'created_at': '2026-04-13T15:07:45.622535446Z', 'done': True, 'done_reason': 'stop', 'total_duration': 997664728, 'load_duration': None, 'prompt_eval_count': 350, 'prompt_eval_duration': None, 'eval_count': 88, 'eval_duration': None, 'logprobs': None, 'model_name': 'qwen3.5:397b-cloud', 'model_provider': 'ollama'}, name='PodTest Agent', id='lc_run--019d8762-5be9-7220-935c-0d1506e25cc7-0', tool_calls=[{'name': 'get_weather_city', 'args': {'city': 'Delhi'}, 'id': 'b381c0f1-c858-41f9-85ce-6ae638d0a94f', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 350, 'output_tokens': 88, 'total_tokens': 438})], structured_response=None)
dynamic_model_selection middleware Done!
ModelResponse(result=[AIMessage(content=''